In [0]:
%pip install 'datacontract-cli[databricks,avro,csv,parquet,sql]'

In [0]:
dbutils.library.restartPython()

In [0]:
import json, yaml
from datacontract.data_contract import DataContract

In [0]:
future_catalog_name = "hive_metastore" # change
future_schema_name = "default" # change

In [0]:
# Read in your ODCS data contract
data_contract_odcs = DataContract(data_contract_file="./odcs_data_contract/odcs_data_contract.yaml", spark=spark).export("odcs")

# Convert the ODCS YAML string to a Python dictionary
data_contract_odcs_yaml = yaml.safe_load(data_contract_odcs)

In [0]:
def get_general_data_quality_rules(table, columns=None):
    """
    Generates a generic set of data quality SQL rules for a given data contract.

    These rules include:
    1. A row count check to ensure the table contains data.
    2. A uniqueness check across specified columns to ensure no duplicate rows.
    Args:
        table (str): The name of the table for which to create rules.
        columns (list, optional): List of column names to use for the duplicate check.
                                  If None or empty, the duplicate rule will be skipped or invalid.
    Returns:
        list: A list of data quality rule dictionaries formatted for use in a data contract.
    """
    partition_by_clause = ", ".join(columns) if columns else ""
    
    general_data_quality_rules = {
        f"{table}": {
            "quality": [
                {
                    "type": "sql",
                    "description": f"Ensures {table} has data",
                    "query": f"SELECT COUNT(*) FROM {table}",
                    "mustBeGreaterThanOrEqualTo": 0
                }
            ]
        }
    }

    # Add duplicate check only if valid columns are provided
    if partition_by_clause:
        general_data_quality_rules[table]["quality"].append(
            {
                "type": "sql",
                "description": f"Ensure {table} has no duplicate rows across all columns",
                "query": f"""
                    SELECT COUNT(*)
                    FROM (
                        SELECT *, COUNT(*) OVER (PARTITION BY {partition_by_clause}) AS row_count
                        FROM {table}
                    ) AS subquery
                    WHERE row_count > 1
                """,
                "mustBe": 0
            }
        )

    # Clean up SQL formatting (flatten multi-line SQL to single-line strings)
    for dq_rule in general_data_quality_rules[table]["quality"]:
        dq_rule["query"] = ' '.join(dq_rule["query"].split())
    return general_data_quality_rules[table]["quality"]

In [0]:
def get_custom_data_quality_rules(table_name):
    """
    Generates a custom set of data quality SQL rules for a given data contract table.
    These rules are specific to the `customer` table and include:
    1. A row count check to ensure the table does not exceed 100 records.
    2. A null check to ensure all customers have an email.
    3. A null check to ensure all customers have both first and last names.
    Args:
        table_name (str): The name of the table for which to generate custom rules.
    Returns:
        list: A list of data quality rule dictionaries formatted for use in a data contract.
    Raises:
        KeyError: If no custom rules are defined for the specified table.
    """
    custom_data_quality_rules = {
        "customer": {
            "quality": [
                {
                    "type": "sql",
                    "description": "Ensures customer table has 100 or less customers",
                    "query": "SELECT COUNT(*) FROM customer",
                    "mustBeLessThanOrEqualTo": 100
                },
                {
                    "type": "sql",
                    "description": "Ensures every customer has an email",
                    "query": "SELECT COUNT(*) FROM customer WHERE email IS NULL",
                    "mustBe": 0
                },
                {
                    "type": "sql",
                    "description": "Ensures every customer has a first and last name",
                    "query": "SELECT COUNT(*) FROM customer WHERE first_name IS NULL OR last_name IS NULL",
                    "mustBe": 0
                }
            ]
        }
    }

    if table_name not in custom_data_quality_rules:
        raise KeyError(f"No custom data quality rules defined for table: {table_name}")
    return custom_data_quality_rules[table_name]["quality"]

In [0]:
def update_data_quality_rules(data_contract, catalog, schema):
    """
    Appends general and custom data quality SQL rules to each table in a data contract schema.
    For each table in the input data contract:
    1. Retrieves the table's columns from Unity Catalog.
    2. Generates general data quality rules based on those columns.
    3. Attempts to retrieve any custom data quality rules (e.g., for specific business checks).
    4. Ensures no duplicate rules are added if they already exist in the contract.
    5. Appends the rules to the table's `quality` section in the data contract.
    Args:
        data_contract (dict): The data contract dictionary in ODCS YAML format.
        catalog (str): The Unity Catalog catalog name where the tables are located.
        schema (str): The Unity Catalog schema name where the tables are located.
    Returns:
        dict: The updated data contract with data quality rules applied to each table.
    """
    for table in data_contract["schema"]:
        if "quality" not in table:
            table["quality"] = []

        # Get column names for the table to generate general DQ rules
        cols = spark.read.table(f"{catalog}.{schema}.{table['name']}").columns
        general_dq_sql_rules = get_general_data_quality_rules(table["name"], cols)

        # Attempt to get custom rules; fall back to general only if not found
        try:
            custom_dq_sql_rules = get_custom_data_quality_rules(table["name"])
            dq_rule_type = "generic and custom"
        except Exception:
            custom_dq_sql_rules = []
            dq_rule_type = "generic"

        # Prevent adding duplicate rules by comparing JSON string representations
        existing_rules = table["quality"]
        set_existing_rules = set(json.dumps(d, sort_keys=True) for d in existing_rules)
        set_general_dq_sql_rules = set(json.dumps(d, sort_keys=True) for d in general_dq_sql_rules)
        set_custom_dq_sql_rules = set(json.dumps(d, sort_keys=True) for d in custom_dq_sql_rules)

        # Check if rules are already applied
        if set_general_dq_sql_rules.issubset(set_existing_rules) and set_custom_dq_sql_rules.issubset(set_existing_rules):
            print(f"already appended '{dq_rule_type}' dq rules to table {table['name']}")
        else:
            table["quality"].extend(general_dq_sql_rules)
            table["quality"].extend(custom_dq_sql_rules)
            print(f"appended '{dq_rule_type}' dq rules to table {table['name']}")

    return data_contract


# Apply data quality rules to the ODCS YAML contract
data_contract_odcs_yaml = update_data_quality_rules(data_contract_odcs_yaml, catalog = future_catalog_name, schema = future_schema_name)